# PDP-CR · So sánh TSR tích hợp phép dời tác vụ từ đầu

Chọn runtime **CPU**, rồi chạy ô benchmark. Ba instance **LC101-n50, LR101-n50 và LRC101-n50** có 50 tác vụ và 10 robot. Mỗi bộ giữ depot và tác vụ ID 1–50 từ CSV của Thảo.

| Phương pháp | Cách tìm kiếm |
|---|---|
| TSR gốc | Reroute, Reverse và Redistribute xuyên suốt lượt chạy |
| TSR hai pha | 90% thời gian chạy TSR gốc, sau đó tinh chỉnh bằng phép gỡ–chèn một hoặc hai tác vụ |
| TSR tích hợp | Thử dời một tác vụ găng ngay từ vòng lặp đầu, cùng với các lân cận gốc |

Mỗi phương pháp dùng cùng nghiệm greedy và cùng seed **0–4**. Năm seed chạy song song trong từng nhóm instance–phương pháp. Mặc định là **600 giây CPU mỗi seed**, tổng cộng 45 lượt. Có thể đặt **TIME_LIMIT = 10** để chạy thử hoặc bỏ chọn **COMPARE_TWO_PHASE** để chỉ so bản tích hợp với TSR gốc.

Notebook tự tải một ZIP khi xong. Trong ZIP, `comparison_summary.csv` so bản tích hợp với TSR gốc, `comparison_summary_vs_two_phase.csv` so với bản hai pha, và `comparison_summary_two_phase.csv` so bản hai pha với TSR gốc. Các tệp `paired_results*.csv` lưu kết quả theo từng seed. Bật **SAVE_TO_DRIVE** để lưu seed đã hoàn thành và chạy tiếp với cùng cấu hình.


In [ ]:
# @title Chạy benchmark PDP-CR
TIME_LIMIT = 600  # @param {type:"number"}
SEEDS = [0, 1, 2, 3, 4]  # @param {type:"raw"}
COMPARE_TWO_PHASE = True  # @param {type:"boolean"}
SAVE_TO_DRIVE = False  # @param {type:"boolean"}

from io import BytesIO
from pathlib import Path
from runpy import run_path
from urllib.request import urlopen
from zipfile import ZipFile

from google.colab import drive, files

REVISION = "2d09ba4091b00095604e1be074dd030f69f9a4e7"
workspace = Path("/content/pdpcr_colab")
source = workspace / f"colab_test-{REVISION}"
runner = source / "run_tsr_colab.py"
if not runner.is_file():
    print("Đang tải mã benchmark từ GitHub...", flush=True)
    url = f"https://codeload.github.com/dhducpro/colab_test/zip/{REVISION}"
    with urlopen(url, timeout=60) as response, ZipFile(BytesIO(response.read())) as archive:
        archive.extractall(workspace)

output_root = workspace / "results"
if SAVE_TO_DRIVE:
    drive.mount("/content/drive")
    output_root = Path("/content/drive/MyDrive/PDPCR_TSR_benchmark")

benchmark = run_path(str(runner))
result_zip = benchmark["run_colab"](
    TIME_LIMIT, SEEDS, benchmark["INTEGRATED"], output_root, REVISION,
    instances=benchmark["N50_INSTANCES"], compare_two_phase=COMPARE_TWO_PHASE,
)
files.download(str(result_zip))
